# AIoT sensors: from InfluxDB to a forecast

1. List the sensors with data
2. Load one field into pandas
3. Flag unusual readings with scikit-learn
4. Forecast the next readings with sktime
5. Ask the gateway's MCP server, as an MCP client

Run the cells from top to bottom (Shift+Enter). `aiot_data.py`, next to this
notebook, does the InfluxDB queries.

In [ ]:
import aiot_data
import matplotlib.pyplot as plt
import pandas as pd

print("InfluxDB:", aiot_data.INFLUX_URL, "database", aiot_data.INFLUX_DB)
devices = aiot_data.sensors("24h")
devices

## 1. Pick a sensor and field

The first sensor that reports `temperature_c` is used, otherwise the first
sensor. To choose yourself, set for example `UID, FIELD = "1:1", "temperature_c"`.
`MAC = None` uses every device of the sensor; set a Device ID to use one.

In [ ]:
with_temp = devices[devices["fields"].map(lambda f: "temperature_c" in f)]
row = (with_temp if len(with_temp) else devices).iloc[0]
UID = row["sensor_uid"]
FIELD = "temperature_c" if "temperature_c" in row["fields"] else row["fields"][0]
MAC = row["mac"]          # None = all devices of UID
WINDOW = "6h"

df = aiot_data.readings(UID, FIELD, MAC, WINDOW)
print(f"{UID} {MAC} {FIELD}: {len(df)} readings in the last {WINDOW}")
df.tail()

In [ ]:
ax = df.plot(x="time", y="value", figsize=(10, 3), legend=False)
ax.set_ylabel(FIELD); ax.set_title(f"{UID} {MAC}")
plt.show()

## 2. Unusual readings (scikit-learn)

An `IsolationForest` learns what normal readings look like, using the value
and its change since the previous reading, and flags the rarest 2 %.

In [ ]:
from sklearn.ensemble import IsolationForest

X = pd.DataFrame({"value": df["value"], "step": df["value"].diff().fillna(0)})
model = IsolationForest(contamination=0.02, random_state=0).fit(X)
df["unusual"] = model.predict(X) == -1
print(df["unusual"].sum(), "unusual readings of", len(df))

ax = df.plot(x="time", y="value", figsize=(10, 3), legend=False)
df[df["unusual"]].plot.scatter(x="time", y="value", color="red", ax=ax)
ax.set_ylabel(FIELD)
plt.show()

## 3. Forecast (sktime)

Readings are averaged per minute, the last 20 % are held back, and two simple
forecasters predict them: a straight-line trend and "same as the last
10 minutes". The mean absolute error says which did better.

In [ ]:
from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.model_selection import temporal_train_test_split
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.trend import PolynomialTrendForecaster
from sktime.performance_metrics.forecasting import mean_absolute_error

y = df.set_index("time")["value"].resample("1min").mean().interpolate()
y.index = y.index.tz_localize(None).to_period("min")
if len(y) < 10:
    print(f"Only {len(y)} minutes of data so far: the forecast needs at least 10. "
          "Wait a few minutes and run this cell again.")
else:
    y_train, y_test = temporal_train_test_split(y, test_size=0.2)
    fh = ForecastingHorizon(y_test.index, is_relative=False)
    w = min(10, len(y_train))
    forecasters = {
        "trend": PolynomialTrendForecaster(degree=1),
        f"last {w} min": NaiveForecaster(strategy="mean", window_length=w),
    }
    ax = y.plot(figsize=(10, 3), label="measured")
    for name, f in forecasters.items():
        pred = f.fit(y_train).predict(fh)
        print(f"{name:12s} mean absolute error {mean_absolute_error(y_test, pred):.3f}")
        pred.plot(ax=ax, label=name)
    ax.set_ylabel(FIELD); ax.legend()
    plt.show()

## 4. MCP client

The same tools Claude uses, called from Python: list the tools, then
`list_sensors` and `read_latest`. In Docker the server is
`http://mcp:8000/mcp` (the `MCP_URL` setting).

In [ ]:
import json
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async def call(tool, **args):
    async with streamable_http_client(aiot_data.MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool, args)
            return json.loads(result.content[0].text) if result.content else None

async def tools():
    async with streamable_http_client(aiot_data.MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return [t.name for t in (await session.list_tools()).tools]

print("tools:", await tools())
print(json.dumps(await call("read_latest", sensor_uid=UID), indent=2))